In [1]:
from pathlib import Path
import numpy as np
import tifffile as tiff


In [5]:

IMG_DIR = Path(r"Z:\Bel\Vascumap_Example_Lifs\labelling\labelled_images\images")
LAB_DIR = Path(r"Z:\Bel\Vascumap_Example_Lifs\labelling\labelled_images\labels")
LABEL_SUFFIX = "_labels"   # adjust if needed


def find_label_for_image(ip: Path, lab_dir: Path, label_suffix: str):
    candidates = [
        lab_dir / f"{ip.stem}{label_suffix}.tif",
        lab_dir / f"{ip.stem}{label_suffix}.tiff",
        lab_dir / f"{ip.stem}{label_suffix}.TIF",
        lab_dir / f"{ip.stem}{label_suffix}.TIFF",
    ]
    lp = next((c for c in candidates if c.exists()), None)
    if lp is None:
        alt = list(lab_dir.glob(f"{ip.stem}{label_suffix}.tif*"))
        lp = alt[0] if alt else None
    return lp


def inspect_tiff(path: Path, kind: str, expect_2d: bool = True):
    """
    Returns: (ok: bool, info: dict)
    Tries to open the TIFF and report potential problems:
      - unreadable / codec missing (e.g., LZW without imagecodecs)
      - non-2D arrays
      - weird dtype
      - NaNs/Infs
      - constant images
      - for masks: not binary-ish
    """
    info = {"path": str(path), "kind": kind}
    try:
        with tiff.TiffFile(str(path)) as tf:
            page0 = tf.pages[0]
            info["compression"] = str(page0.compression)
            info["dtype_page0"] = str(page0.dtype)
            info["shape_page0"] = tuple(page0.shape)
            info["is_ome"] = bool(tf.is_ome)
            info["is_imagej"] = bool(getattr(tf, "is_imagej", False))
    except Exception as e:
        info["error_open"] = f"{type(e).__name__}: {e}"
        return False, info

    # Now try to actually read pixel data
    try:
        arr = np.squeeze(tiff.imread(str(path)))
    except Exception as e:
        info["error_read"] = f"{type(e).__name__}: {e}"
        return False, info

    info["ndim"] = int(arr.ndim)
    info["shape"] = tuple(arr.shape)
    info["dtype"] = str(arr.dtype)

    if expect_2d and arr.ndim != 2:
        info["issue_dim"] = f"Expected 2D, got ndim={arr.ndim}, shape={arr.shape}"

    # Basic numeric checks
    if np.issubdtype(arr.dtype, np.floating):
        finite = np.isfinite(arr)
        info["finite_frac"] = float(finite.mean()) if arr.size else 0.0
        if not finite.all():
            info["issue_nonfinite"] = "Contains NaN/Inf"

    # Range / const checks
    try:
        vmin = float(np.min(arr)) if arr.size else float("nan")
        vmax = float(np.max(arr)) if arr.size else float("nan")
        info["min"] = vmin
        info["max"] = vmax
        if arr.size and vmin == vmax:
            info["issue_constant"] = "All pixels have identical value"
    except Exception as e:
        info["issue_minmax"] = f"Could not compute min/max: {type(e).__name__}: {e}"

    # Mask-specific checks (binary-ish)
    if kind == "mask" and arr.size:
        uniq = np.unique(arr)
        info["unique_count"] = int(len(uniq))
        info["unique_preview"] = uniq[:10].tolist()

        # common acceptable: {0,1}, {0,255}, boolean
        if arr.dtype == np.bool_:
            pass
        else:
            # if many unique values, it might not be a mask
            if len(uniq) > 10:
                info["issue_not_binary"] = f"Mask has many unique values (count={len(uniq)})."
            else:
                # if values not close to 0/1/255, flag
                allowed = {0, 1, 255}
                if not set(int(x) for x in uniq).issubset(allowed):
                    info["issue_unexpected_values"] = f"Mask values not in {sorted(allowed)}: {uniq.tolist()}"

        # foreground fraction
        try:
            fg = (arr > 0).mean()
            info["fg_frac_(>0)"] = float(fg)
            if fg < 1e-4:
                info["issue_empty_mask"] = "Mask is almost entirely background"
            if fg > 0.999:
                info["issue_full_mask"] = "Mask is almost entirely foreground"
        except Exception as e:
            info["issue_fg_frac"] = f"Could not compute fg fraction: {type(e).__name__}: {e}"

    return True, info


def audit_dataset(img_dir: Path, lab_dir: Path, label_suffix: str):
    img_paths = sorted(img_dir.glob("*.tif*"))
    report = {
        "missing_label": [],
        "image_fail": [],
        "mask_fail": [],
        "warnings": [],
        "ok": [],
    }

    for ip in img_paths:
        lp = find_label_for_image(ip, lab_dir, label_suffix)
        if lp is None:
            report["missing_label"].append(str(ip))
            continue

        img_ok, img_info = inspect_tiff(ip, "image", expect_2d=True)
        msk_ok, msk_info = inspect_tiff(lp, "mask", expect_2d=True)

        # Collect hard failures
        if not img_ok:
            report["image_fail"].append(img_info)
            print("Image name failure {}".format(str(ip)))
        if not msk_ok:
            report["mask_fail"].append(msk_info)
            print("Mask name failure {}".format(str(ip)))


        # Collect warnings (issues but still readable)
        issues = []
        for d in (img_info, msk_info):
            for k in list(d.keys()):
                if k.startswith("issue_"):
                    issues.append((d["kind"], k, d[k], d["path"]))

        if issues:
            report["warnings"].append({
                "image": str(ip),
                "mask": str(lp),
                "issues": issues,
                "img_shape": img_info.get("shape"),
                "msk_shape": msk_info.get("shape"),
                "img_comp": img_info.get("compression"),
                "msk_comp": msk_info.get("compression"),
            })
        else:
            report["ok"].append({"image": str(ip), "mask": str(lp)})

    return report


def print_summary(report):
    print("=== DATASET AUDIT SUMMARY ===")
    print(f"OK pairs:            {len(report['ok'])}")
    print(f"Missing labels:      {len(report['missing_label'])}")
    print(f"Image read failures: {len(report['image_fail'])}")
    print(f"Mask read failures:  {len(report['mask_fail'])}")
    print(f"Warnings:            {len(report['warnings'])}")
    print()

    if report["missing_label"]:
        print("---- Missing labels (first 10) ----")
        for p in report["missing_label"][:10]:
            print(p)
        print()

    if report["image_fail"]:
        print("---- Image failures (first 5) ----")
        for d in report["image_fail"][:5]:
            print(d.get("path"), "->", d.get("error_open") or d.get("error_read"))
        print()

    if report["mask_fail"]:
        print("---- Mask failures (first 5) ----")
        for d in report["mask_fail"][:5]:
            print(d.get("path"), "->", d.get("error_open") or d.get("error_read"))
        print()

    if report["warnings"]:
        print("---- Warnings (first 10) ----")
        for w in report["warnings"][:10]:
            print("Image:", w["image"])
            print("Mask: ", w["mask"])
            for kind, key, msg, path in w["issues"]:
                print(f"  - [{kind}] {key}: {msg}")
            print()


# Run audit
report = audit_dataset(IMG_DIR, LAB_DIR, LABEL_SUFFIX)
print_summary(report)

=== DATASET AUDIT SUMMARY ===
OK pairs:            73
Missing labels:      0
Image read failures: 0
Mask read failures:  0
Warnings:            0

